In [ ]:
# 第9周-Day6：Domain Model Diagram — 对象关系、生命周期、依赖
# matplotlib 中文字体配置
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

## 📅 Week 9 - Day 6 | 2026-08-01（周六·动手日）

### ⚡ Domain Model Diagram — 对象关系、生命周期、依赖

> **📌 今日核心问题：哪个对象最可能被合并？哪个最可能被拆分？**

| 项目 | 内容 |
|---|---|
| **周主题** | Domain Deep Dive |
| **今日交付物** | 一张完整的 Domain Model 关系图 + 生命周期图 + 合并/拆分分析 |
| **本周已拆对象** | BlueprintVersion, SkillRelease, Deployment/Revision, ReleaseChannel/TrafficPolicy, DigitalEmployeeDefinition |

## ❓ 今日核心问题

### 哪个对象最可能被合并？哪个最可能被拆分？

这是架构师的核心判断力。不是"对象越多越好"，也不是"对象越少越好"。

> **每个对象必须能回答一个问题——"如果把它合并到另一个对象里，会失去什么？如果拆不出来，会造成什么耦合？"**

判定原则（从 ADR-003 提炼）：
> 如果两个对象的**演化节奏不同、生命周期不同、变更 Owner 不同**，它们就不应该合并。

In [ ]:
# LangChat v2 四层架构全图
fig, ax = plt.subplots(figsize=(16, 12))
ax.axis('off')
ax.set_xlim(-0.5, 16)
ax.set_ylim(-1, 13)

# Layer backgrounds
layers = [
    (9.5, 12.5, '#E8F5E9', '#2E7D32', 'Business Domain Layer'),
    (6.0, 9.5, '#FFF3E0', '#E65100', 'Supply Chain Layer'),
    (2.0, 6.0, '#E3F2FD', '#0D47A1', 'Runtime Layer'),
    (-0.5, 2.0, '#F3E5F5', '#4A148C', 'Operations Layer'),
]

for y_bot, y_top, color, edge, label in layers:
    rect = mpatches.FancyBboxPatch((0, y_bot), 15, y_top - y_bot,
                                    boxstyle="round,pad=0.1", facecolor=color, edgecolor=edge, linewidth=2, alpha=0.6)
    ax.add_patch(rect)
    ax.text(0.3, y_top - 0.3, label, fontsize=11, fontweight='bold', color=edge, va='top')

# Business Domain objects
bd_objects = [
    (2, 11.5, 'DigitalEmployee\nDefinition\n(BD-01)', '#C8E6C9'),
    (6, 11.5, 'ApplicationContract\n(BD-02/03)', '#C8E6C9'),
    (10, 11.5, 'Capability\n(SC-07)', '#C8E6C9'),
    (12.5, 11.5, 'KnowledgeCollection\n(SC-09)', '#C8E6C9'),
    (2, 10.0, 'Policy\n(SC-11)', '#C8E6C9'),
]
for x, y, label, color in bd_objects:
    rect = mpatches.FancyBboxPatch((x-1, y-0.5), 2.2, 1, boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='#2E7D32', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x+0.1, y, label, ha='center', va='center', fontsize=7.5, fontweight='bold')

# Supply Chain objects
sc_objects = [
    (1.5, 8.5, 'BlueprintCandidate\n(SC-02)'),
    (4, 8.5, 'BlueprintVersion\n(SC-03) ★'),
    (6.5, 8.5, 'Build/BuildRun\n(SC-04/05)'),
    (9, 8.5, 'ExecutionPlanIR\n(SC-06)'),
    (11.5, 8.5, 'SkillRelease v2\n(SC-13) ★★'),
    (14, 8.5, 'ReleaseChannel\n(SC-14)'),
    (1.5, 7.0, 'CapabilityRelease\n(SC-08)'),
    (4, 7.0, 'KnowledgeSnapshot\n(SC-10)'),
    (6.5, 7.0, 'PolicyBundle\n(SC-12)'),
    (9, 7.0, 'PromotionEvent\n(SC-15)'),
    (11.5, 7.0, 'ReleaseEvaluation\n(SC-19)'),
]
for x, y, label in sc_objects:
    color = '#FFE0B2' if '★' not in label else '#FFB74D'
    rect = mpatches.FancyBboxPatch((x-1, y-0.4), 2.2, 0.9, boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='#E65100', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x+0.1, y, label.replace(' ★','').replace(' ★★',''), ha='center', va='center', fontsize=7, fontweight='bold')
    if '★★' in label:
        ax.text(x+1.3, y+0.5, '★唯一可部署', fontsize=6, color='#BF360C', fontweight='bold')
    elif '★' in label:
        ax.text(x+1.1, y+0.5, '★canonical', fontsize=6, color='#BF360C')

# Runtime objects
rt_objects = [
    (1.5, 5.0, 'Deployment\n(RT-01)'),
    (4.5, 5.0, 'DeploymentRevision\n(RT-02) ★闭包'),
    (8, 5.0, 'TrafficPolicy\n(RT-03)'),
    (11, 5.0, 'FrozenExecContext\n(RT-04)'),
    (14, 5.0, 'RuntimeABI\n(RT-05)'),
    (3, 3.5, 'Execution\n(RT-06)'),
    (6, 3.5, 'Session\n(RT-07)'),
    (8.5, 3.5, 'State\n(RT-08)'),
    (11, 3.5, 'Memory\n(RT-09)'),
    (14, 3.5, 'DeploymentEval\n(RT-10)'),
]
for x, y, label in rt_objects:
    rect = mpatches.FancyBboxPatch((x-1, y-0.4), 2.2, 0.9, boxstyle="round,pad=0.1",
                                    facecolor='#BBDEFB', edgecolor='#0D47A1', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x+0.1, y, label, ha='center', va='center', fontsize=7, fontweight='bold')

# Operations objects
ops_objects = [
    (2.5, 1.0, 'Registry\n(SC-16)'),
    (6, 1.0, 'Catalog Projection\n(SC-17)'),
    (9.5, 1.0, 'Attestation\n(GX-01)'),
    (12.5, 1.0, 'Provenance\n(GX-02)'),
]
for x, y, label in ops_objects:
    rect = mpatches.FancyBboxPatch((x-1, y-0.4), 2.2, 0.9, boxstyle="round,pad=0.1",
                                    facecolor='#E1BEE7', edgecolor='#4A148C', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x+0.1, y, label, ha='center', va='center', fontsize=7, fontweight='bold')

# Arrows: artifact chain (key flows)
arrows = [
    (2.5, 8.5, 3.0, 8.5, '评审'),
    (5.0, 8.5, 5.5, 8.5, '编译'),
    (7.5, 8.5, 8.0, 8.5, '产出'),
    (10.0, 8.5, 10.5, 8.5, '打包'),
    (12.5, 8.5, 13.0, 8.5, '晋升'),
    (5.5, 5.0, 3.5, 5.0, '物化'),  # SkillRelease -> DeploymentRevision
    (5.5, 5.0, 7.5, 5.0, '路由'),  # Revision -> TrafficPolicy
]
for x1, y1, x2, y2, label in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#B71C1C', lw=2))
    ax.text((x1+x2)/2, (y1+y2)/2+0.15, label, fontsize=6, color='#B71C1C', ha='center')

ax.set_title('LangChat v2 四层架构 — 35个Domain Object全景图', fontsize=15, fontweight='bold', pad=20)
ax.text(7.5, -0.8, '制品链流向严格单向：Candidate → Version → Build → IR → SkillRelease → DeploymentRevision → TrafficPolicy → Execution',
        ha='center', fontsize=9, color='#B71C1C', style='italic')

plt.tight_layout()
plt.show()

## 📋 ADR 依据：合并/拆分硬约束

### 绝不可合并（违反 Charter §6.4 "同一对象不跨层归属"）

| 两个对象 | 理由 |
|---|---|
| SkillRelease ↔ DeploymentRevision | Supply Chain ≠ Runtime |
| ReleaseChannel ↔ TrafficPolicy | Channel 是 Supply Chain，TrafficPolicy 是 Runtime |
| BlueprintVersion ↔ ExecutionPlanIR | 源制品 ≠ 中间表示 |
| DigitalEmployeeDefinition ↔ Deployment | 定义层 ≠ 运行层 |

### 有条件可合并（但会失去关键属性）

| 对象对 | 合并代价 |
|---|---|
| Capability + CapabilityRelease | 失去版本化 |
| KnowledgeCollection + KnowledgeSnapshot | 失去快照不可变性 |
| Policy + PolicyBundle | 失去批量管理 |

### 判定框架：三不同原则

| 对象对 | 演化节奏 | 生命周期 | 变更 Owner | 合并？ |
|---|---|---|---|---|
| Capability / CapabilityRelease | 慢 vs 快 | 不同 | 不同团队 | ⚠️ 可合并 |
| Collection / Snapshot | 可变 vs 冻结 | 完全不同 | 不同团队 | ⚠️ 可合并 |
| Policy / PolicyBundle | 单条 vs 批量 | 完全不同 | 不同团队 | ⚠️ 可合并 |

In [ ]:
# 代码 vs 目标态覆盖度
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

categories = [
    ('Business Domain', [40, 60, 100], '#4CAF50'),
    ('Supply Chain', [60, 40, 100], '#FF9800'),
    ('Runtime', [10, 90, 100], '#2196F3'),
    ('Operations', [30, 70, 100], '#9C27B0'),
]

for ax, (name, vals, color) in zip(axes, categories):
    labels = ['已实现', '缺失', '目标态']
    wedges, texts, autotexts = ax.pie(vals, labels=labels, autopct='%1.0f%%',
                                       colors=[color, '#E0E0E0', '#F5F5F5'],
                                       startangle=90, textprops={'fontsize': 9})
    ax.set_title(f'{name}\n覆盖度 {vals[0]}%', fontsize=12, fontweight='bold')

fig.suptitle('LangChat v2 四层代码覆盖度', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n关键发现：Runtime Layer 覆盖度仅 10%")
print("4个核心对象（Deployment, DeploymentRevision, TrafficPolicy, FrozenExecutionContext）完全不存在")
print("所有 v2 治理目前都是'纸面架构'

## 🔍 代码验证：对象实现状态

### 已实现 ✅
- BlueprintCandidate + Admission + Review → SC-02
- BlueprintVersion (frozen=True + SHA-256) → SC-03
- SkillReleaseRegistry (v1版) → SC-13 v1
- DigitalEmployeeModel (v1版) → BD-01 v1
- Observability (span/trace/emitter) → 横切

### 部分实现 🟡
- Build/BuildRun (stub) → SC-04/05
- SkillRelease (v1 tag-based, 非 v2 OCI) → SC-13
- RuntimeABI (散落在 canonical/) → RT-05

### 完全缺失 ❌
- Deployment → RT-01
- DeploymentRevision → RT-02
- TrafficPolicy → RT-03
- FrozenExecutionContext → RT-04
- ReleaseChannel → SC-14
- CapabilityRelease → SC-08
- KnowledgeSnapshot → SC-10
- PolicyBundle → SC-12
- ReleaseEvaluation → SC-19

In [ ]:
# MI CRE：10个Mall的数字员工体系
fig, ax = plt.subplots(figsize=(14, 7))
ax.axis('off')
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)

# Definition layer
rect = mpatches.FancyBboxPatch((3, 6.5), 8, 1.2, boxstyle="round,pad=0.15",
                                facecolor='#C8E6C9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(rect)
ax.text(7, 7.4, 'DigitalEmployeeDefinition: "合同审核数字员工 - 小合" v1.0', ha='center', va='center', fontsize=10, fontweight='bold')
ax.text(7, 6.9, '引用: ApplicationContractVersion + BlueprintVersion | scope: [shanghai, beijing, guangzhou, ...7个]', ha='center', fontsize=8, color='#555')

# Deployments
malls = ['上海 Mall', '北京 Mall', '广州 Mall', '... 7个', '成都 Mall']
mall_colors = ['#E3F2FD', '#FFEBEE', '#E8F5E9', '#FFF3E0', '#F3E5F5']
statuses = ['✅ Active', '⏸ Suspended', '✅ Active', '✅ Active', '✅ Active']
x_positions = [1.5, 4, 6.5, 9, 11.5]

for i, (mall, color, status, x) in enumerate(zip(malls, mall_colors, statuses, x_positions)):
    rect = mpatches.FancyBboxPatch((x-1, 3.5), 2.2, 2.5, boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='#666', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x+0.1, 5.5, mall, ha='center', fontsize=9, fontweight='bold')
    ax.text(x+0.1, 5.0, f'Deployment-{chr(65+i)}', ha='center', fontsize=7.5, color='#555')
    ax.text(x+0.1, 4.5, status, ha='center', fontsize=8)
    ax.text(x+0.1, 3.8, '独立 Revision\n独立 TrafficPolicy\n独立 kill_switch', ha='center', fontsize=6.5, color='#777')

    # Arrow from definition
    ax.annotate('', xy=(x+0.1, 6.0), xytext=(7, 6.5),
                arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.5, alpha=0.5))

ax.text(7, 2.5, '同一个定义 → 10个独立部署\n每个 Mall 有自己的知识快照、灰度策略、紧急停止',
        ha='center', fontsize=10, color='#B71C1C', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#FFF9C4', edgecolor='#F57F17'))

ax.text(7, 0.8, '如果合并 Definition 和 Deployment（错误做法）：\n• 10个Mall变成1个，无法独立管理\n• 上海暂停 = 全部暂停\n• 广州更新知识库 = 全部变更',
        ha='center', fontsize=8, color='#D32F2F',
        bbox=dict(boxstyle='round', facecolor='#FFEBEE', edgecolor='#D32F2F'))

ax.set_title('MI CRE：同一个数字员工定义 → 10个独立部署', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 📊 传统单体 vs Dify vs LangChat

| 维度 | 传统 ERP | Dify/LangChain | LangChat v2 |
|---|---|---|---|
| 对象数量 | 少（一张表） | 中（5个） | 多（35+） |
| 关系复杂度 | 低 | 中 | 高（四层引用链） |
| 版本管理 | 数据库审计 | git | digest + 版本内建 |
| 回滚 | 覆盖旧文件 | git revert | 前向回滚 |
| 灰度 | 不支持 | 不原生 | TrafficPolicy 内建 |
| 审计 | 日志 | git log | PromotionEvent + Attestation |

**为什么 LangChat 选了最复杂的路？** 因为面向企业级生产环境。个人工具可以牺牲治理换易用性，企业工具必须牺牲易用性换治理。

## 🧠 架构师思考题

**场景A**：如果团队只有 3 个人，哪些对象可以合并？
- Capability + CapabilityRelease → ⚠️ 可短期合并，留拆分接口
- Deployment + DeploymentRevision → ❌ 不建议（灰度需要多 Revision）
- BlueprintVersion + ExecutionPlanIR → ❌ 绝不建议（违反 HC-3）

**场景B**：如果未来要支持"多模型供应商"，哪些对象需要拆分？
> 提示：答案藏在 SkillRelease 的依赖锁里。

**场景C**：Serverless 数字员工（按需启动），Deployment 还需要吗？
> 提示：DeploymentRevision 闭包仍然需要，但 Deployment 生命周期语义可能演化。

## 💡 我的理解变化 + Week 9 总结

| 以前以为 | 现在知道 |
|---|---|
| Domain Model = ER图 + 类图 | Domain Model 是**治理拓扑图** |
| 对象越多 = 越复杂 | 对象多≠复杂，35个各司其职比5个什么塞都好 |
| 合并 = 简化 | 合并 = **增加耦合** |
| Diagram 就是文档 | Diagram 是**架构决策的推演工具** |

### Week 9 五个对象的理解深度

| 对象 | 深度(1-10) | 关键发现 |
|---|---|---|
| BlueprintVersion | 8.5 | 三层不可变保证 |
| SkillRelease | 8.0 | 治理决定非技术决定 |
| Deployment/Revision | 7.5 | 回滚是前向操作 |
| ReleaseChannel/TrafficPolicy | 7.5 | Supply Chain ≠ Runtime 完全解耦 |
| DigitalEmployeeDefinition | 7.0 | 定义是语义锚点，kill_switch 跨层 |

## 🔗 明日连接

**Day7（周日）：🔄 Virtual CTO Review** — ADR Health Check + 五维评分 + 进度报告